In [5]:
import nibabel as nib
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider


file_path = "C:\\Users\\shivg\\Downloads\\MRI\\UCSF_BrainMetastases_TRAIN\\100413A\\100413A_T1post.nii.gz"
nifti_img = nib.load(file_path)
image_data = nifti_img.get_fdata()

def show_slice(slice_index):
    # 'slice_index' will be controlled by the slider
    
    # .T transposes the slice for a more standard "radiological" view
    # 'origin="lower"' places the (0,0) coordinate at the bottom-left
    plt.imshow(image_data[:, :, slice_index].T, cmap='gray', origin='lower')
    plt.title(f'Axial Slice: {slice_index}')
    plt.axis('off')
    plt.show()


# We assume the 3rd axis (index 2) is the one we want to slice
max_slice = image_data.shape[2] - 1
interact(show_slice, 
         slice_index=IntSlider(min=0, max=max_slice, step=1, value=max_slice // 2));

interactive(children=(IntSlider(value=53, description='slice_index', max=107), Output()), _dom_classes=('widge…

In [8]:
import pydicom
import matplotlib.pyplot as plt
import numpy as np
import os
from ipywidgets import interact, IntSlider

# --- 1. Define your DICOM directory ---
folder_path = "C:\\Users\\shivg\\Downloads\\dicom_series"

def load_dicom_series(folder_path):
    """
    Loads a DICOM series from a folder, sorts slices by position,
    and stacks them into a 3D NumPy array.
    """
    slices = []
    # Find all .dcm files in the directory
    for f in os.listdir(folder_path):
        file_path = os.path.join(folder_path, f)
        if os.path.isfile(file_path) and f.endswith('.dcm'):
            try:
                ds = pydicom.dcmread(file_path)
                # Check if it's an image file
                if 'PixelData' in ds:
                    slices.append(ds)
            except Exception as e:
                print(f"Warning: Could not read {file_path}: {e}")

    if not slices:
        print("Error: No DICOM files with PixelData found in folder.")
        return None

    # Sort the slices by their 'ImagePositionPatient' (Z-coordinate)
    # This is crucial for correct 3D reconstruction
    slices.sort(key=lambda x: float(x.ImagePositionPatient[2]))

    # Stack the slices into a 3D array
    # We get the shape from the first slice
    image_shape = slices[0].pixel_array.shape
    image_3d = np.zeros((image_shape[0], image_shape[1], len(slices)))
    
    for i, s in enumerate(slices):
        image_3d[:, :, i] = s.pixel_array
        
    return image_3d

# --- 2. Load the 3D volume ---
print(f"Loading DICOM series from: {folder_path}")
image_volume = load_dicom_series(folder_path)

if image_volume is not None:
    print(f"Loaded volume with shape: {image_volume.shape}")

    # --- 3. Create the interactive plotting function ---
    def show_axial_slice(slice_index):
        plt.imshow(image_volume[:, :, slice_index].T, cmap='gray', origin='lower')
        plt.title(f'Axial Slice: {slice_index}')
        plt.axis('off')
        plt.show()

    # --- 4. Create the slider ---
    max_slice = image_volume.shape[2] - 1
    interact(show_axial_slice, 
             slice_index=IntSlider(min=0, max=max_slice, step=1, value=max_slice // 2));


# --- This code assumes you have 'image_volume' from Option 2 ---

# Make sure you have the 'image_volume' variable loaded!
if 'image_volume' not in locals():
    print("Error: Please run the code from Option 2 first to load 'image_volume'")
else:
    # Get the dimensions for the sliders
    shape = image_volume.shape
    x_max, y_max, z_max = shape[0] - 1, shape[1] - 1, shape[2] - 1

    # --- Create the interactive plotting function ---
    def plot_orthogonal_views(x, y, z):
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        
        # 1. Sagittal (slicing along X)
        axes[0].imshow(image_volume[x, :, :].T, cmap='gray', origin='lower', aspect='auto')
        axes[0].set_title(f'Sagittal (X = {x})')
        axes[0].set_xlabel('Y')
        axes[0].set_ylabel('Z')

        # 2. Coronal (slicing along Y)
        axes[1].imshow(image_volume[:, y, :].T, cmap='gray', origin='lower', aspect='auto')
        axes[1].set_title(f'Coronal (Y = {y})')
        axes[1].set_xlabel('X')
        axes[1].set_ylabel('Z')

        # 3. Axial (slicing along Z)
        axes[2].imshow(image_volume[:, :, z].T, cmap='gray', origin='lower', aspect='auto')
        axes[2].set_title(f'Axial (Z = {z})')
        axes[2].set_xlabel('X')
        axes[2].set_ylabel('Y')
        
        plt.tight_layout()
        plt.show()

    # --- Create Sliders for each dimension ---
    interact(
        plot_orthogonal_views, 
        x=IntSlider(min=0, max=x_max, step=1, value=x_max // 2, description='X (Sagittal):'),
        y=IntSlider(min=0, max=y_max, step=1, value=y_max // 2, description='Y (Coronal):'),
        z=IntSlider(min=0, max=z_max, step=1, value=z_max // 2, description='Z (Axial):')
    );

Loading DICOM series from: C:\Users\shivg\Downloads\dicom_series
Loaded volume with shape: (256, 256, 108)


interactive(children=(IntSlider(value=53, description='slice_index', max=107), Output()), _dom_classes=('widge…

interactive(children=(IntSlider(value=127, description='X (Sagittal):', max=255), IntSlider(value=127, descrip…